## Vector database and Faiss Index

In [ ]:
!pip install faiss-gpu

In [ ]:
import faiss
import numpy as np
import polars as pl
import torch
import time
import torch.nn as nn

# ==========================================
# 1. THE TWO-TOWER ARCHITECTURE (FULL FEATURE SET)
# ==========================================

class UserTower(nn.Module):
    # num_dense_features = 25 (7 new features + 18 onehot features)
    def __init__(self, num_items, item_embed_dim=32, num_dense_features=25, final_dim=64):
        super().__init__()
        self.item_embedding = nn.Embedding(num_embeddings=num_items, embedding_dim=item_embed_dim, padding_idx=0)
        self.gru = nn.GRU(input_size=item_embed_dim, hidden_size=64, batch_first=True)
        self.static_mlp = nn.Sequential(nn.Linear(num_dense_features, 32), nn.ReLU())
        self.fusion_layer = nn.Linear(64 + 32, final_dim)

    def forward(self, history_seq, static_features):
        seq_emb = self.item_embedding(history_seq)
        _, hidden = self.gru(seq_emb)
        user_history_vector = hidden.squeeze(0) 
        user_static_vector = self.static_mlp(static_features) 
        combined = torch.cat([user_history_vector, user_static_vector], dim=1)
        return self.fusion_layer(combined)


class ItemTower(nn.Module):
    def __init__(self, num_items, num_categories=2000, num_tags=500, item_embed_dim=32, nlp_embed_dim=512, final_dim=64):
        super().__init__()
        self.item_embedding = nn.Embedding(num_items, item_embed_dim, padding_idx=0)
        
        # Categorical & Multi-Hot Tag Embeddings
        self.cat1_embedding = nn.Embedding(num_categories, 8, padding_idx=0)
        self.cat2_embedding = nn.Embedding(num_categories, 8, padding_idx=0)
        self.tag_embedding  = nn.Embedding(num_tags, 8, padding_idx=0)
        
        self.text_mlp = nn.Sequential(
            nn.Linear(nlp_embed_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 32)
        )
        
        # Fusion Input: 
        # 32 (Item ID) + 8 (Cat1) + 8 (Cat2) + 8 (Pooled Tags) + 32 (Text MLP) + 1 (Log Duration) = 89
        self.fusion_layer = nn.Linear(32 + 8 + 8 + 8 + 32 + 1, final_dim)

    def forward(self, item_id, cat1_id, cat2_id, tags, duration_log, precomputed_text_vector):
        id_emb = self.item_embedding(item_id)
        c1_emb = self.cat1_embedding(cat1_id)
        c2_emb = self.cat2_embedding(cat2_id)
        
        # Multi-hot Bag-of-Tags Pooling:
        # tags shape: (Batch, MAX_TAGS) -> tag_emb: (Batch, MAX_TAGS, 8)
        tag_emb = self.tag_embedding(tags)
        # Sum across MAX_TAGS dimension -> shape: (Batch, 8)
        tag_emb = tag_emb.sum(dim=1) 
        
        text_feat = self.text_mlp(precomputed_text_vector)
        dur_feat = duration_log.unsqueeze(1) 
        
        combined = torch.cat([id_emb, c1_emb, c2_emb, tag_emb, text_feat, dur_feat], dim=1)
        return self.fusion_layer(combined)


class TwoTowerModel(nn.Module):
    def __init__(self, num_items, num_categories, num_tags, final_dim=64):
        super().__init__()
        self.user_tower = UserTower(num_items=num_items, final_dim=final_dim)
        self.item_tower = ItemTower(
            num_items=num_items, 
            num_categories=num_categories, 
            num_tags=num_tags, 
            final_dim=final_dim
        )

    def forward(self, history_seq, user_static, item_id, cat1_id, cat2_id, tags, duration_log, item_features):
        u_vector = self.user_tower(history_seq, user_static)
        i_vector = self.item_tower(item_id, cat1_id, cat2_id, tags, duration_log, item_features)
        
        u_vector = torch.nn.functional.normalize(u_vector, p=2, dim=1)  
        i_vector = torch.nn.functional.normalize(i_vector, p=2, dim=1)  
        
        similarity_matrix = torch.matmul(u_vector, i_vector.T)
        return similarity_matrix * 10.0

# ==========================================
# 2. FAISS AND RETRIEVAL LOGIC
# ==========================================

def build_faiss_index_gpu(df_item_embeddings, vector_dim=64):
    print("⚡ Building GPU FAISS Vector Index...")
    video_ids_map = df_item_embeddings["video_id"].to_numpy().astype(np.int64)
    item_vectors = np.array(df_item_embeddings["item_embedding_64d"].to_list(), dtype=np.float32)

    faiss.normalize_L2(item_vectors)

    cpu_index = faiss.IndexFlatIP(vector_dim)
    cpu_index.add(item_vectors)

    res = faiss.StandardGpuResources()
    gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)

    print(f"✅ GPU FAISS Index Built! Total Indexed Items: {gpu_index.ntotal:,}")
    return gpu_index, video_ids_map


def retrieve_top_k_faiss(model, target_user_row, faiss_index, video_ids_map, device, k=200):
    model.eval()

    # 1. Compute dynamic User Vector (Shape: 1, 64)
    raw_history = target_user_row.get("history_sequence", [])
    if raw_history is None:
        raw_history = []
    MAX_SEQ_LEN = 100
    padded_history = ([0] * MAX_SEQ_LEN + raw_history)[-MAX_SEQ_LEN:]
    history_seq = torch.tensor([padded_history], dtype=torch.long).to(device)

    # --- FIX: Replicate the 25-feature string-to-float mapping logic ---
    activity_map = {"unknown": 0.0, "low_active": 1.0, "high_active": 2.0, "full_active": 3.0}
    follow_map = {'0': 0.0, '(0,10]': 1.0, '(10,50]': 2.0, '(50,100]': 3.0, '(100,150]': 4.0, '(150,250]': 5.0, '(250,500]': 6.0, '500+': 7.0}
    fans_map = {'0': 0.0, '[1,10)': 1.0, '[10,100)': 2.0, '[100,1k)': 3.0, '[1k,5k)': 4.0, '[5k,1w)': 5.0, '[1w,10w)': 6.0}
    friend_map = {'0': 0.0, '[1,5)': 1.0, '[5,30)': 2.0, '[30,60)': 3.0, '[60,120)': 4.0, '[120,250)': 5.0, '250+': 6.0}
    register_map = {'15-30': 1.0, '31-60': 2.0, '61-90': 3.0, '91-180': 4.0, '181-365': 5.0, '366-730': 6.0, '730+': 7.0}

    def safe_map(val, mapping_dict):
        return float(mapping_dict.get(str(val), 0.0))

    user_static_list = [
        safe_map(target_user_row.get("user_active_degree"), activity_map),
        float(target_user_row.get("is_live_streamer") or 0.0),
        float(target_user_row.get("is_video_author") or 0.0),
        safe_map(target_user_row.get("follow_user_num_range"), follow_map),
        safe_map(target_user_row.get("fans_user_num_range"), fans_map),
        safe_map(target_user_row.get("friend_user_num_range"), friend_map),
        safe_map(target_user_row.get("register_days_range"), register_map)
    ] + [float(target_user_row.get(f"onehot_feat{i}") or 0.0) for i in range(18)]

    # Cast to Tensor (Shape: 1, 25)
    user_static = torch.tensor([user_static_list], dtype=torch.float32).to(device)

    # Perform Forward Pass
    with torch.no_grad():
        user_vector = model.user_tower(history_seq, user_static)
        user_vector = torch.nn.functional.normalize(user_vector, p=2, dim=1)
        user_vector_np = user_vector.cpu().numpy().astype(np.float32)

    # 2. Query GPU FAISS Index
    scores, retrieved_indices = faiss_index.search(user_vector_np, k)

    # 3. Map array indices back to real video_ids
    top_k_items = video_ids_map[retrieved_indices[0]].tolist()

    return top_k_items

# ==========================================
# 3. EXECUTION PIPELINE
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Running inference on device: {device}")

# 1. Load Item Embeddings & Build GPU FAISS Index
df_item_faiss = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/faiss-item-embeddings/kaggle/working/faiss_item_embeddings.parquet")
faiss_gpu_index, video_ids_map = build_faiss_index_gpu(df_item_faiss, vector_dim=64)

# 2. Load User Data
df_users = pl.read_parquet("/kaggle/input/datasets/nguyenngocanhle/two-tower-data/user_table.parquet")
test_user = df_users.to_dicts()[0]

# 3. Load Checkpoint
checkpoint_path = "/kaggle/input/models/nguyenngocanhle/kuairand-1k-two-tower/pytorch/default/4/kaggle/working/best_two_tower_inbatch.pth"
checkpoint = torch.load(checkpoint_path, map_location="cpu")

state_dict = (
    checkpoint["model_state_dict"]
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint
    else checkpoint
)

# --- FIX: Dynamically detect all vocabulary sizes from saved weights ---
num_items = state_dict["user_tower.item_embedding.weight"].shape[0]
num_categories = state_dict["item_tower.cat1_embedding.weight"].shape[0]
num_tags = state_dict["item_tower.tag_embedding.weight"].shape[0]

print(f"📦 Extracted from weights -> Items: {num_items}, Categories: {num_categories}, Tags: {num_tags}")

model = TwoTowerModel(
    num_items=num_items, 
    num_categories=num_categories, 
    num_tags=num_tags, 
    final_dim=64
)

model.load_state_dict(state_dict)
model = model.to(device)

# Warmup GPU
# with torch.no_grad():
#     _ = retrieve_top_k_faiss(model, test_user, faiss_gpu_index, video_ids_map, device, k=10)

# 4. Perform Benchmark
start_time = time.time()
recommended_videos = retrieve_top_k_faiss(
    model=model,
    target_user_row=test_user,
    faiss_index=faiss_gpu_index,
    video_ids_map=video_ids_map,
    device=device,
    k=200,
)
elapsed = (time.time() - start_time) * 1000

print(f"⏱️ GPU Retrieval Time: {elapsed:.2f} ms")
print(f"🎬 Top 5 Recommended Video IDs: {recommended_videos[:5]}")